# Train CNN Notebook

Versao em notebook de `train_cnn.py`, separada em celulas para facilitar ajustes no codigo e nos hiperparametros.

In [ ]:
import csv
import os
from dataclasses import dataclass
from typing import Dict

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms

from dataset import SimpleFrameDataset

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


@dataclass
class Metrics:
    loss: float
    accuracy: float
    precision: float
    recall: float
    f1_score: float
    auc: float
    fpr: float


@dataclass
class EpochResult:
    metrics: Metrics
    labels: list
    preds: list

In [ ]:
def build_transforms(image_size):
    return transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02),
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


def build_eval_transforms(image_size):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


def compute_metrics(total_loss, total_samples, labels, preds, positive_scores):
    loss = total_loss / total_samples if total_samples else 0.0
    accuracy = float(accuracy_score(labels, preds))
    precision = float(precision_score(labels, preds, zero_division=0))
    recall = float(recall_score(labels, preds, zero_division=0))
    f1 = float(f1_score(labels, preds, zero_division=0))
    try:
        auc = float(roc_auc_score(labels, positive_scores))
    except ValueError:
        auc = float('nan')

    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    return Metrics(
        loss=loss,
        accuracy=accuracy,
        precision=precision,
        recall=recall,
        f1_score=f1,
        auc=auc,
        fpr=fpr,
    )


def write_metrics_table(output_path, rows):
    fieldnames = ['epoch', 'split', 'loss', 'accuracy', 'precision', 'recall', 'f1_score', 'auc', 'fpr']
    with open(output_path, 'w', newline='', encoding='utf-8') as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def save_loss_curve(output_path, train_losses, test_losses):
    plt.figure(figsize=(8, 5))
    epochs = range(1, len(train_losses) + 1)
    plt.plot(epochs, train_losses, marker='o', label='train loss')
    plt.plot(epochs, test_losses, marker='o', label='test loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train/Test Loss Curve')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()


def save_confusion_matrix(output_path, labels, preds):
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.colorbar()
    tick_labels = ['negative', 'positive']
    tick_marks = [0, 1]
    plt.xticks(tick_marks, tick_labels)
    plt.yticks(tick_marks, tick_labels)
    plt.xlabel('Predicted label')
    plt.ylabel('True label')

    threshold = cm.max() / 2 if cm.size else 0.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = 'white' if cm[i, j] > threshold else 'black'
            plt.text(j, i, str(cm[i, j]), ha='center', va='center', color=color)

    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()


def get_class_distribution(dataset) -> Dict[int, int]:
    distribution = {0: 0, 1: 0}
    for _, _, label, _ in dataset.samples:
        distribution[label] = distribution.get(label, 0) + 1
    return distribution


def build_weighted_sampler(dataset):
    class_distribution = get_class_distribution(dataset)
    sample_weights = []
    for _, _, label, _ in dataset.samples:
        class_count = class_distribution[label]
        sample_weights.append(1.0 / class_count if class_count else 0.0)
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )


def build_class_weights(dataset, device):
    class_distribution = get_class_distribution(dataset)
    total = sum(class_distribution.values())
    weights = []
    for label in [0, 1]:
        class_count = class_distribution.get(label, 0)
        weights.append(total / (2 * class_count) if class_count else 0.0)
    return torch.tensor(weights, dtype=torch.float32, device=device)


def run_epoch(model, loader, criterion, device, threshold=0.5, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_samples = 0
    labels_list = []
    preds_list = []
    positive_scores = []

    for images, labels, _, _ in loader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = (probs >= threshold).long()
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size
        labels_list.extend(labels.detach().cpu().tolist())
        preds_list.extend(preds.detach().cpu().tolist())
        positive_scores.extend(probs.detach().cpu().tolist())

    metrics = compute_metrics(total_loss, total_samples, labels_list, preds_list, positive_scores)
    return EpochResult(metrics=metrics, labels=labels_list, preds=preds_list)

In [ ]:
# Ajuste estes valores antes de executar o treino.
ROOT = 'path/to/dataset/root'
EPOCHS = 5
BATCH_SIZE = 16
NUM_WORKERS = 2
LR = 1e-3
IMAGE_SIZE = 128
FRAME_STRIDE = 15
MAX_FRAMES_PER_VIDEO = 24
TRAIN_VIDEO_LIMIT = None
TEST_VIDEO_LIMIT = None
OUTPUT_DIR = 'runs/simple_cnn_notebook'
DECISION_THRESHOLD = 0.5
SELECTION_METRIC = 'f1'
DISABLE_CLASS_WEIGHTS = False
DISABLE_WEIGHTED_SAMPLER = False

assert SELECTION_METRIC in {'accuracy', 'f1', 'recall', 'auc'}
os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('output dir:', OUTPUT_DIR)

In [ ]:
train_transform = build_transforms(IMAGE_SIZE)
eval_transform = build_eval_transforms(IMAGE_SIZE)

train_dataset = SimpleFrameDataset(
    root=ROOT,
    mode='train',
    transform=train_transform,
    frame_stride=FRAME_STRIDE,
    max_frames_per_video=MAX_FRAMES_PER_VIDEO,
    video_limit=TRAIN_VIDEO_LIMIT,
)
test_dataset = SimpleFrameDataset(
    root=ROOT,
    mode='test',
    transform=eval_transform,
    frame_stride=FRAME_STRIDE,
    max_frames_per_video=MAX_FRAMES_PER_VIDEO,
    video_limit=TEST_VIDEO_LIMIT,
)

train_distribution = get_class_distribution(train_dataset)
test_distribution = get_class_distribution(test_dataset)
train_sampler = None if DISABLE_WEIGHTED_SAMPLER else build_weighted_sampler(train_dataset)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=train_sampler is None,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print('train samples:', len(train_dataset))
print('test samples:', len(test_dataset))
print('train class distribution:', train_distribution)
print('test class distribution:', test_distribution)

In [ ]:
model = SimpleCNN().to(device)
class_weights = None if DISABLE_CLASS_WEIGHTS else build_class_weights(train_dataset, device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print('decision threshold:', DECISION_THRESHOLD)
print('selection metric:', SELECTION_METRIC)
print('weighted sampler:', not DISABLE_WEIGHTED_SAMPLER)
print('class weights:', None if class_weights is None else class_weights.detach().cpu().tolist())

In [ ]:
best_score = float('-inf')
metrics_rows = []
train_losses = []
test_losses = []

metrics_path = os.path.join(OUTPUT_DIR, 'metrics_table.csv')
loss_curve_path = os.path.join(OUTPUT_DIR, 'loss_curve.png')
confusion_matrix_path = os.path.join(OUTPUT_DIR, 'confusion_matrix.png')

for epoch in range(1, EPOCHS + 1):
    train_result = run_epoch(
        model,
        train_loader,
        criterion,
        device,
        threshold=DECISION_THRESHOLD,
        optimizer=optimizer,
    )
    eval_result = run_epoch(
        model,
        test_loader,
        criterion,
        device,
        threshold=DECISION_THRESHOLD,
    )

    train_metrics = train_result.metrics
    eval_metrics = eval_result.metrics
    train_losses.append(train_metrics.loss)
    test_losses.append(eval_metrics.loss)

    metrics_rows.extend([
        {
            'epoch': epoch,
            'split': 'train',
            'loss': train_metrics.loss,
            'accuracy': train_metrics.accuracy,
            'precision': train_metrics.precision,
            'recall': train_metrics.recall,
            'f1_score': train_metrics.f1_score,
            'auc': train_metrics.auc,
            'fpr': train_metrics.fpr,
        },
        {
            'epoch': epoch,
            'split': 'test',
            'loss': eval_metrics.loss,
            'accuracy': eval_metrics.accuracy,
            'precision': eval_metrics.precision,
            'recall': eval_metrics.recall,
            'f1_score': eval_metrics.f1_score,
            'auc': eval_metrics.auc,
            'fpr': eval_metrics.fpr,
        },
    ])

    write_metrics_table(metrics_path, metrics_rows)
    save_loss_curve(loss_curve_path, train_losses, test_losses)
    save_confusion_matrix(confusion_matrix_path, eval_result.labels, eval_result.preds)

    print(
        'epoch {}/{} | train loss {:.4f} acc {:.4f} prec {:.4f} rec {:.4f} f1 {:.4f} auc {:.4f} fpr {:.4f} | '
        'test loss {:.4f} acc {:.4f} prec {:.4f} rec {:.4f} f1 {:.4f} auc {:.4f} fpr {:.4f}'.format(
            epoch,
            EPOCHS,
            train_metrics.loss,
            train_metrics.accuracy,
            train_metrics.precision,
            train_metrics.recall,
            train_metrics.f1_score,
            train_metrics.auc,
            train_metrics.fpr,
            eval_metrics.loss,
            eval_metrics.accuracy,
            eval_metrics.precision,
            eval_metrics.recall,
            eval_metrics.f1_score,
            eval_metrics.auc,
            eval_metrics.fpr,
        )
    )

    selection_score = getattr(eval_metrics, SELECTION_METRIC)
    if selection_score > best_score:
        best_score = selection_score
        torch.save(
            {
                'model_state_dict': model.state_dict(),
                'config': {
                    'root': ROOT,
                    'epochs': EPOCHS,
                    'batch_size': BATCH_SIZE,
                    'num_workers': NUM_WORKERS,
                    'lr': LR,
                    'image_size': IMAGE_SIZE,
                    'frame_stride': FRAME_STRIDE,
                    'max_frames_per_video': MAX_FRAMES_PER_VIDEO,
                    'train_video_limit': TRAIN_VIDEO_LIMIT,
                    'test_video_limit': TEST_VIDEO_LIMIT,
                    'output_dir': OUTPUT_DIR,
                    'decision_threshold': DECISION_THRESHOLD,
                    'selection_metric': SELECTION_METRIC,
                    'disable_class_weights': DISABLE_CLASS_WEIGHTS,
                    'disable_weighted_sampler': DISABLE_WEIGHTED_SAMPLER,
                },
                f'best_{SELECTION_METRIC}_score': best_score,
            },
            os.path.join(OUTPUT_DIR, 'best.pt'),
        )

print('metrics table saved to:', metrics_path)
print('loss curve saved to:', loss_curve_path)
print('confusion matrix saved to:', confusion_matrix_path)
print('best score:', best_score)

In [ ]:
from IPython.display import Image, display

display(Image(filename=loss_curve_path))
display(Image(filename=confusion_matrix_path))